# Imports and packages

In [1]:
## helpful packages

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import random
import re
import string

## nltk imports
import nltk
### uncomment and run these lines if you haven't downloaded relevant nltk add-ons yet
#nltk.download('averaged_perceptron_tagger')
#nltk.download('stopwords')
from nltk import pos_tag
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords

## spacy imports
import spacy
### uncomment and run the below line if you haven't loaded the en_core_web_sm library yet
#! python3 -m spacy download en_core_web_sm
import en_core_web_sm
nlp = en_core_web_sm.load()

## vectorizer
from sklearn.feature_extraction.text import CountVectorizer

## sentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

## lda
from gensim import corpora
import gensim

## repeated printouts and wide-format text
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
pd.set_option('display.max_colwidth', None)

# Text analysis of Department of Justice (DOJ) press releases (50 points total)

- For background:

    - DOJ is the federal law enforcement agency responsible for federal prosecutions; this contrasts with the local prosecutions in the Cook County dataset we analyzed earlier. Here's a short explainer on which crimes get prosecuted federally versus locally: https://www.criminaldefenselawyer.com/resources/criminal-defense/federal-crime/state-vs-federal-crimes.htm#:~:text=Federal%20criminal%20prosecutions%20are%20handled,of%20state%20and%20local%20law. 
    - Here's the Kaggle that contains the data: https://www.kaggle.com/jbencina/department-of-justice-20092018-press-releases 

In [2]:
## first, unzip the file
## then, run this code to load the unzipped json file and convert to a dataframe
## and convert some of the attributes from lists to values
doj = pd.read_json("../datasets/combined.json", lines = True)

## due to json, topics are in a list so remove them and concatenate with ;
doj['topics_clean'] = ["; ".join(topic) 
                      if len(topic) > 0 else "No topic" 
                      for topic in doj.topics]

## similarly with components
doj['components_clean'] = ["; ".join(comp) 
                           if len(comp) > 0 else "No component" 
                           for comp in doj.components]

## drop older columns from data
doj = doj[['id', 'title', 'contents', 'date', 'topics_clean', 
           'components_clean']].copy()

doj.head()

id  \
0     None   
1  12-919    
2  11-1002   
3   10-015   
4   18-898   

                                                                                                          title  \
0                                                                  Convicted Bomb Plotter Sentenced to 30 Years   
1                              $1 Million in Restitution Payments Announced to Preserve North Carolina Wetlands   
2                 $1 Million Settlement Reached for Natural Resource Damages at Superfund Site in Massachusetts   
3                                          10 Las Vegas Men Indicted \r\nfor Falsifying Vehicle Emissions Tests   
4  $100 Million Settlement Will Speed Cleanup Work at Centredale Manor Superfund Site in North Providence, R.I.   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

## 1. Tagging and sentiment scoring (16 points)

Focus on the following press release: `id` == "17-1204" about this pharmaceutical kickback prosecution: https://www.forbes.com/sites/michelatindera/2017/11/16/fentanyl-billionaire-john-kapoor-to-plead-not-guilty-in-opioid-kickback-case/?sh=21b8574d6c6c 

The `contents` column is the one we're treating as a document. You may need to to convert it from a pandas series to a single string.

We'll call the raw string of this press release `pharma`

In [3]:
## your code to subset to one press release and take the string

In [4]:
pharma = doj.contents[doj.id == "17-1204"].iloc[0]




### 1.1 part of speech tagging (3 points)

A. Preprocess the `pharma` press release to remove all punctuation / digits (so can use `.isalpha()` to subset)

B. With the preprocessed press release from part A, use the part of speech tagger within nltk to tag all the words in that one press release with their part of speech. 

C. Using the output from B, extract the adjectives and sort those adjectives from most occurrences to fewest occurrences. Print a dataframe with the 5 most frequent adjectives and their counts in the `pharma` release. See here for a list of the names of adjectives within nltk: https://pythonprogramming.net/natural-language-toolkit-nltk-part-speech-tagging/

**Resources**:

- Documentation for .isalpha(): https://www.w3schools.com/python/ref_string_isalpha.asp




In [5]:
## your code here to restrict to alpha

In [6]:
## your code here for part of speech tagging

In [7]:
## remove punct
one_release_alpha_word = [word for word in wordpunct_tokenize(pharma) if 
                    word.isalpha()]
one_release_alpha_word[0:5]

['The', 'founder', 'and', 'majority', 'owner']

In [8]:
## use nltk for part of speech tagging
## try part of speech tagging using nltk
tokens_pos = pos_tag(one_release_alpha_word) # generate part of speech tags for those tokens

## iterate over the tuple and extract all adjectives
tokens_adj = [one_tok[0] for one_tok in tokens_pos
               if one_tok[1] in ['JJ', 'JJJ', 'JJS']]

pd.DataFrame(tokens_adj).value_counts().head(5)

0         
former        8
opioid        5
nationwide    4
addictive     3
other         3
Name: count, dtype: int64

## 1.2 named entity recognition (3 points)



A. Using the original `pharma` press release (so the one before stripping punctuation/digits), use spaCy to extract all named entities from the press release.

B. Print the unique named entities with the tag: `LAW`. Here's some background on what RICO means: https://www.justia.com/criminal/docs/rico/ 

In [9]:
## your code here 

In [10]:
## extract and print named entities
spacy_r = nlp(pharma)
law_ent = [one_tok.text for one_tok in spacy_r.ents
          if one_tok.label_ == "LAW"]
set(law_ent)


{'RICO', 'the Controlled Substances Act'}

C. You want to extract the possible sentence lengths the CEO is facing; pull out the named entities with (1) the label `DATE` and (2) that contain the word year or years. Print these named entities.

**Hint:**  
You may want to use the `re` module for the second part.

In [11]:
## your code here

In [12]:
## get the dates
all_dates_wyear = [one_tok.text for one_tok in spacy_r.ents
            if one_tok.label_ == "DATE" and 
            re.search(r"year(s)?", one_tok.text) is not None]

all_dates_wyear


['last year', '20 years', 'three years', 'five years', 'three years']

D. Parse the pharma string at the sentence level. Note that this involves more than just splitting on each `.`; for full credit, add at least one additional delimiter that marks the end of the sentence.

Then, using those sentences, pull and print the original sentences from the press releases where those year lengths are mentioned. Describe in your own words (1 sentence) what length of sentence (prison) and probation (supervised release) the CEO may be facing if convicted after this indictment (if there are multiple lengths mentioned describe the maximum). 

**Hint:**  
You may want to use re.search or re.findall 



In [13]:
## your code here

In [14]:
## parse original string at sentence level and find those that contain the sentence
one_release_split = re.split('\.\\xa0|\. |\.”|\.', pharma)

rel_splits = [sen for sen in one_release_split 
             if re.search("|".join(all_dates_wyear), sen) is not None]
rel_splits

['"More than 20,000 Americans died of synthetic opioid overdoses last year, and millions are addicted to opioids',
 'The charges of conspiracy to commit RICO and conspiracy to commit mail and wire fraud each provide for a sentence of no greater than 20 years in prison, three years of supervised release and a fine of $250,000, or twice the amount of pecuniary gain or loss',
 ' The charges of conspiracy to violate the Anti-Kickback Law provide for a sentence of no greater than five years in prison, three years of supervised release and a $25,000 fine']

## 1.3 sentiment analysis  (10 points)


A. Subset the press releases to those labeled with one of three topics via `topics_clean`: Civil Rights, Hate Crimes, and Project Safe Childhood. We'll call this `doj_subset` going forward and it should have 717 rows.



In [15]:
## your code here for subsetting

In [16]:
doj_subset = doj[doj.topics_clean.isin(["Hate Crimes", 
                            "Civil Rights", "Project Safe Childhood"])].copy()
doj_subset.shape

(717, 6)

B. Write a function that takes one press release string as an input and:

- Removes named entities from each press release string (**Hint:** you may want to use `re.sub` with an or condition)
- Scores the sentiment of the entire press release using the `SentimentIntensityAnalyzer` and `polarity_scores`
- Returns the length-four (negative, positive, neutral, compound) sentiment dictionary (any order is fine)

Apply that function to each of the press releases in `doj_subset`. 

**Hints**: 

- I used a function + list comprehension to execute and it takes about 30 seconds on my local machine; if it's taking a very long time, you may want to check your code for inefficiencies. If you can't fix those, for partial credit on this part/full credit on remainder, you can take a small random sample of the 717


In [17]:
def removene_score(one_release):
    
    ## remove named entities
    spacy_ents = nlp(one_release).ents
    spacy_ents_text = [re.sub(punct_exceptspace,
                          "", 
                        ent.text) for ent in spacy_ents]
    release_nopunct = re.sub(punct_exceptspace, "", one_release)
    string_noents = re.sub("|".join(spacy_ents_text),
                           "", release_nopunct)
    
    ## check removal
    assert len(one_release) != len(string_noents)
    
    ## score sentiment
    sentiment_all = sent_obj.polarity_scores(string_noents)
    
    ## return sentiment dict
    return(sentiment_all)
    
    

In [18]:
## initialize sentiment classifier 
sent_obj = SentimentIntensityAnalyzer()

## define pattern of punc without space
punct_exceptspace = r"[{}]".format(string.punctuation.replace("\s", 
                            ""))

all_dicts = [removene_score(one_release) for one_release 
             in doj_subset.contents]

C. Add the four sentiment scores to the `doj_subset` dataframe to create a dataframe: `doj_subset_wscore`. Sort from highest neg to lowest neg score (so most negative to least negative) and print the `id`, `contents`, and `neg` columns of the two most negative press releases. 

Notes:

- Don't worry if your sentiment score differs slightly from our output on GitHub; changes in preprocessing can lead to diff scores

In [19]:
all_dicts_df = pd.DataFrame(all_dicts)

## add to main data
doj_subset_wscore = pd.concat([doj_subset.reset_index(), 
                        all_dicts_df.reset_index()], axis = 1)
doj_subset_wscore[['id', 'contents', 'neg']].sort_values(by = 'neg', 
                ascending = False).head(2)

,id,contents,neg
13,14-248,"The Department of Justice announced that this morning John W. Ng, 58, of Albuquerque, N.M., made his initial appearance in federal court on a criminal complaint charging him with a hate crime offense. This charge is related to anti-Semitic threats Ng made against a Jewish woman who owns and operates the Nosh Jewish Delicatessen and Bakery in Albuquerque. Ng was arrested by the FBI on March 7, 2014, based on a criminal complaint alleging that he interfered with the victim’s federally protected rights by threatening her and interfering with her business because of her religion. According to the criminal complaint, between Jan. 22, 2014, and Feb. 8, 2014, Ng allegedly posted threatening anti-Semitic notes on and in the vicinity of the victim’s business. A criminal complaint merely establishes probable cause, and Ng is presumed innocent unless proven guilty. If convicted on the offense charged in the criminal complaint, Ng faces a maximum statutory penalty of one year in prison. This matter was investigated by the Albuquerque Division of the FBI and is being prosecuted by Assistant U.S. Attorney Mark T. Baker of the U.S. Attorney’s Office for the District of New Mexico and Trial Attorney AeJean Cha of the U.S. Department of Justice’s Civil Rights Division.",0.350
632,16-718,"In a nine-count indictment unsealed today, two Mississippi correctional officers were charged with beating an inmate and a third was charged with helping to cover it up. The indictment charged Lawardrick Marsher, 28, and Robert Sturdivant, 47, officers at Mississippi State Penitentiary, in Parchman, Mississippi, with a beating that included kicking, punching and throwing the victim to the ground. Marsher and Sturdivant were charged with violating the right of K.H., a convicted prisoner, to be free from cruel and unusual punishment. Sturdivant was also charged with failing to intervene while Marsher was punching and beating K.H. The indictment alleges that their actions involved the use of a dangerous weapon and resulted in bodily injury to the victim. A third officer, Deonte Pate, 23, was charged along with Marsher and Sturdivant for conspiring to cover up the beating. The indictment alleges that all three officers submitted false reports and that all three lied to the FBI. If convicted, Marsher and Sturdivant face a maximum sentence of 10 years in prison on the excessive force charges. Each of the three officers faces up to five years in prison on the conspiracy and false statement charges, and up to 20 years in prison on the false report charges. An indictment is merely an accusation, and the defendants are presumed innocent unless and until proven guilty. This case is being investigated by the FBI’s Jackson Division, with the cooperation of the Mississippi Department of Corrections. It is being prosecuted by Assistant U.S. Attorney Robert Coleman of the Northern District of Mississippi and Trial Attorney Dana Mulhauser of the Civil Rights Division’s Criminal Section. Marsher Indictment",0.313


D. With the dataframe from part C, find the mean compound sentiment score for each of the three topics in `topics_clean` using group_by and agg.

E. Add a 1 sentence interpretation of why we might see the variation in scores (remember that compound is a standardized summary where -1 is most negative; +1 is most positive)


In [20]:
## agg and find the mean compound score by topic
doj_subset_wscore.groupby(['topics_clean']).agg({'compound': np.mean})

,compound
topics_clean,
Civil Rights,-0.088875
Hate Crimes,-0.936014
Project Safe Childhood,-0.644885


# 2. Topic modeling (25 points)

For this question, use the `doj_subset_wscores` data that is restricted to civil rights, hate crimes, and project safe childhood and with the sentiment scores added


## 2.1 Preprocess the data by removing stopwords, punctuation, and non-alpha words (5 points)

A. Write a function that:

- Takes in a single raw string in the `contents` column from that dataframe
- Does the following preprocessing steps:

    - Converts the words to lowercase
    - Removes stopwords, adding the custom stopwords in the code cell below to the default stopwords list
    - Only retains alpha words (so removes digits and punctuation)
    - Only retains words 4 characters or longer
    - Uses the snowball stemmer from nltk to stem

- Returns a joined preprocessed string (so if press release is something like "The CEO was indicted" it might return "ceo indict" 
    
B. Use `apply` or list comprehension to execute that function and create a new column in the data called `processed_text`. Note: there will be a deduction if your code uses a non-list comprehension for loop that uses append.
    
C. Print the `id`, `contents`, and `processed_text` columns for the following press releases:

id = 16-718 (this case: https://www.seattletimes.com/nation-world/doj-miami-police-reach-settlement-in-civil-rights-case/)

id = 16-217 (this case: https://www.wlbt.com/story/32275512/three-mississippi-correctional-officers-indicted-for-inmate-assault-and-cover-up/)
    

In [21]:
custom_doj_stopwords = ["civil", "rights", "division", "department", "justice",
                        "office", "attorney", "district", "case", "investigation", "assistant",
                       "trial", "assistance", "assist"]

In [22]:
def processtext(row, colname, stopword_list, stemmer, min_token_length):
    
    ## get string form of listing
    string_of_col = str(row[colname])
    try:
        ## convert to lower
        string_low = string_of_col.lower()
        #print(string_low)
        
        ## remove stopwords 
        remove_stop = [word for word in wordpunct_tokenize(string_low)
                      if word not in stopword_list]
        processed_string = " ".join([stemmer.stem(i) 
                        for i in remove_stop if 
                        i.isalpha() and len(i) >= min_token_length])
        return(processed_string)
    except:
        processed_string = "" # to handle data errors where not actually text
        return(processed_string)

In [23]:
## initialize snowball stemmer and call list of stopwords
snow_stemmer = SnowballStemmer(language='english')
list_stopwords = stopwords.words("english") + custom_doj_stopwords

## execute 
doj_subset_wscore['processed_text'] = doj_subset_wscore.apply(processtext,
                             axis = 1,
                             args = ["contents", list_stopwords, snow_stemmer, 4])


In [24]:
## print examples
doj_subset_wscore.loc[doj_subset_wscore.id == "16-718",
                     ['id', 'contents', 'processed_text']]

doj_subset_wscore.loc[doj_subset_wscore.id == "16-217",
                     ['id', 'contents', 'processed_text']]


,id,contents,processed_text
632,16-718,"In a nine-count indictment unsealed today, two Mississippi correctional officers were charged with beating an inmate and a third was charged with helping to cover it up. The indictment charged Lawardrick Marsher, 28, and Robert Sturdivant, 47, officers at Mississippi State Penitentiary, in Parchman, Mississippi, with a beating that included kicking, punching and throwing the victim to the ground. Marsher and Sturdivant were charged with violating the right of K.H., a convicted prisoner, to be free from cruel and unusual punishment. Sturdivant was also charged with failing to intervene while Marsher was punching and beating K.H. The indictment alleges that their actions involved the use of a dangerous weapon and resulted in bodily injury to the victim. A third officer, Deonte Pate, 23, was charged along with Marsher and Sturdivant for conspiring to cover up the beating. The indictment alleges that all three officers submitted false reports and that all three lied to the FBI. If convicted, Marsher and Sturdivant face a maximum sentence of 10 years in prison on the excessive force charges. Each of the three officers faces up to five years in prison on the conspiracy and false statement charges, and up to 20 years in prison on the false report charges. An indictment is merely an accusation, and the defendants are presumed innocent unless and until proven guilty. This case is being investigated by the FBI’s Jackson Division, with the cooperation of the Mississippi Department of Corrections. It is being prosecuted by Assistant U.S. Attorney Robert Coleman of the Northern District of Mississippi and Trial Attorney Dana Mulhauser of the Civil Rights Division’s Criminal Section. Marsher Indictment",nine count indict unseal today mississippi correct offic charg beat inmat third charg help cover indict charg lawardrick marsher robert sturdiv offic mississippi state penitentiari parchman mississippi beat includ kick punch throw victim ground marsher sturdiv charg violat right convict prison free cruel unusu punish sturdiv also charg fail interven marsher punch beat indict alleg action involv danger weapon result bodili injuri victim third offic deont pate charg along marsher sturdiv conspir cover beat indict alleg three offic submit fals report three lie convict marsher sturdiv face maximum sentenc year prison excess forc charg three offic face five year prison conspiraci fals statement charg year prison fals report charg indict mere accus defend presum innoc unless proven guilti investig jackson cooper mississippi correct prosecut robert coleman northern mississippi dana mulhaus crimin section marsher indict


,id,contents,processed_text
313,16-217,"The Justice Department has reached a comprehensive settlement agreement with the city of Miami and the Miami Police Department (MPD) resolving the Justice Department’s investigation of officer-involved shootings by MPD officers, announced Principal Deputy Assistant Attorney General Vanita Gupta, head of the Justice Department’s Civil Rights Division and U.S. Attorney Wifredo A. Ferrer of the Southern District of Florida. The settlement, which was approved by Miami’s city commission today and will go into effect when the agreement is signed by all parties, resolves claims stemming from the Justice Department’s investigation into officer-involved shootings by MPD officers, which was conducted under the Violent Crime Control and Law Enforcement Act of 1994. The investigation’s findings, issued in July 2013, identified a pattern or practice of excessive use of force through officer-involved shootings in violation of the Fourth Amendment of the Constitution. The city’s compliance with the settlement will be monitored by an independent reviewer, former Tampa, Florida, Police Chief Jane Castor. Under the settlement agreement, the city will implement comprehensive reforms to ensure constitutional policing and support public trust. The settlement agreement is designed to minimize officer-involved shootings and to more effectively and quickly investigate officer-involved shootings that do occur, through measures that include: “This settlement represents a renewed commitment by the city of Miami and Chief Rodolfo Llanes to provide constitutional policing for Miami residents and to protect public safety through sustainable reform,” said Principal Deputy Assistant Attorney General Gupta. “The agreement will help to strengthen the relationship between the MPD and the communities they serve by improving accountability for officers who fire their weapons unlawfully, and provides for community participation in the enforcement of this agreement.” “Today's agreement is the result of a joint effort between the Department of Justice and the City of Miami to ensure that the Miami Police Department continues its efforts to make our community safe while protecting the sacred Constitutional rights of all of our citizens,” said U.S. Attorney Ferrer. “Through oversight and communication, the agreement seeks to make permanent the positive changes that former Chief Orosa and Chief Llanes have made, and we applaud the City Commission’s vote.” The settlement agreement builds upon important reforms implemented by the city since the Justice Department issued its findings, including: The investigation was conducted by attorneys and staff from the Civil Rights Division’s Special Litigation Section and the Civil Division of the U. S. Attorney’s Office of the Southern District of Florida.",reach comprehens settlement agreement citi miami miami polic resolv offic involv shoot offic announc princip deputi general vanita gupta head wifredo ferrer southern florida settlement approv miami citi commiss today effect agreement sign parti resolv claim stem offic involv shoot offic conduct violent crime control enforc find issu juli identifi pattern practic excess forc offic involv shoot violat fourth amend constitut citi complianc settlement monitor independ review former tampa florida polic chief jane castor settlement agreement citi implement comprehens reform ensur constitut polic support public trust settlement agreement design minim offic involv shoot effect quick investig offic involv shoot occur measur includ settlement repres renew commit citi miami chief rodolfo llane provid constitut polic miami resid protect public safeti sustain reform said princip deputi general gupta agreement help strengthen relationship communiti serv improv account offic fire weapon unlaw provid communiti particip enforc agreement today agreement result joint effort citi miami ensur miami polic continu effort make communiti safe protect sacr constit

## 2.2 Create a document-term matrix from the preprocessed press releases and to explore top words (5 points)

A. Use the `create_dtm` function I provide (alternately, feel free to write your own!) and create a document-term matrix using the preprocessed press releases; make sure metadata contains the following columns: `id`, `compound` sentiment column you added, and the `topics_clean` column

B. Print the top 10 words for press releases with compound sentiment in the top 5% (so the most positive sentiment)

C. Print the top 10 words for press releases with compound sentiment in the bottom 5% (so the most negative sentiment)

**Hint**: for these, remember the pandas quantile function from pset two.  

D. Print the top 10 words for press releases in each of the three `topics_clean`

For steps B - D, to receive full credit, write a function `get_topwords` that helps you avoid duplicated code when you find top words for the different subsets of the data. There are different ways to structure it but one way is to feed it subsetted data (so data subsetted to one topic etc.) and for it to get the top words for that subset.



In [25]:
def create_dtm(list_of_strings, metadata):
    vectorizer = CountVectorizer(lowercase = True)
    dtm_sparse = vectorizer.fit_transform(list_of_strings)
    dtm_dense_named = pd.DataFrame(dtm_sparse.todense(), 
        columns=vectorizer.get_feature_names())
    dtm_dense_named_withid = pd.concat([metadata.reset_index(), dtm_dense_named], axis = 1)
    return(dtm_dense_named_withid)

In [26]:
def get_topw(subsetted_data):
    
    ## return the top 10 words for term cols in subsetted data
    top_w = subsetted_data[[col for col in subsetted_data.columns if "raw" 
                            not in col and "index" not in col]].sum(axis =
                                0).sort_values(ascending = 
                                False).head(10)
    
    ## return
    return(top_w)

In [27]:
## A. 
dtm = create_dtm(doj_subset_wscore.processed_text,
                doj_subset_wscore[['id', 'topics_clean', 'compound']].rename(columns = 
                            {'id': 'id_raw',
                            'topics_clean': 'topics_clean_raw',
                            'compound': 'compound_raw'}))

## B. 
### top 5% lowest sentiment
dtm_mostneg = dtm[dtm.compound_raw <= dtm.compound_raw.quantile(0.05)].copy()
dtm_mostneg.shape
print("Top words in the lowest 5% compound score press releases")
get_topw(dtm_mostneg)

###  top 5% highest sentiment
dtm_mostpos = dtm[dtm.compound_raw >= dtm.compound_raw.quantile(0.95)].copy()
dtm_mostpos.shape
print("Top words in the highest 5% compound score press releases")
get_topw(dtm_mostpos)


## C. 
for topic in dtm.topics_clean_raw.unique():

    subset = dtm[dtm.topics_clean_raw == topic]
    print("For topic: " + topic)
    get_topw(subset)


(36, 6870)

Top words in the lowest 5% compound score press releases


assault     181
crime       172
victim      164
offic       155
hate        129
defend      123
sentenc     104
conspir     104
american     95
feder        95
dtype: int64

(36, 6870)

Top words in the highest 5% compound score press releases


agreement     173
enforc        130
state         118
ensur         110
disabl        107
communiti      99
settlement     87
student        87
general        84
servic         83
dtype: int64

... (2 outputs left out)

For topic: Project Safe Childhood


child          1022
exploit         701
sexual          572
safe            479
childhood       474
project         472
pornographi     452
children        423
crimin          405
prosecut        374
dtype: int64

For topic: Hate Crimes


victim      591
crime       557
hate        524
defend      484
prosecut    478
charg       463
sentenc     455
american    451
feder       432
guilti      430
dtype: int64

## 2.3 Estimate a topic model using those preprocessed words (5 points)

A. Going back to the preprocessed words from part 2.1, estimate a topic model with 3 topics, since you want to see if the unsupervised topic models recover different themes for each of the three manually-labeled topics (civil rights; hate crimes; project safe childhood). You have free rein over the other topic model parameters beyond the number of topics.

B. After estimating the topic model, print the top 15 words in each topic.

**Hints and Resources**:

- Same topic modeling resources linked to above
- Make sure to use the `random_state` argument within the model so that the numbering of topics does not move around between runs of your code

In [28]:
## tokenize text
tokenized_text = [wordpunct_tokenize(one_pr)
                  for one_pr in doj_subset_wscore.processed_text]

In [29]:
## preprocess and estimate topicmod
### create dictionary
text_proc_dict = corpora.Dictionary(tokenized_text)

### filter dictionary- using 5% as bounds
text_proc_dict.filter_extremes(no_below = round(doj_subset_wscore.shape[0]*0.05),
                             no_above = round(doj_subset_wscore.shape[0]*0.95))

### create corpus from dictionary
corpus_fromdict_proc = [text_proc_dict.doc2bow(one_text) 
                   for one_text in tokenized_text]

### estimate model
n_topics = 3
ldamod_proc = gensim.models.ldamodel.LdaModel(corpus_fromdict_proc, 
                                         num_topics = n_topics, id2word=text_proc_dict, 
                                         passes=20, alpha = 'auto',
                                        per_word_topics = True, random_state = 91988)

In [30]:
### print topics and words
topics = ldamod_proc.print_topics(num_words = 15)
for topic in topics:
    print(topic)

(0, '0.040*"child" + 0.026*"exploit" + 0.024*"sexual" + 0.018*"safe" + 0.018*"project" + 0.017*"childhood" + 0.017*"pornographi" + 0.016*"children" + 0.015*"crimin" + 0.014*"prosecut" + 0.013*"sentenc" + 0.013*"victim" + 0.012*"minor" + 0.012*"ceo" + 0.011*"year"')
(1, '0.015*"hous" + 0.014*"victim" + 0.013*"defend" + 0.012*"said" + 0.012*"crime" + 0.012*"charg" + 0.012*"feder" + 0.011*"hate" + 0.011*"prosecut" + 0.011*"american" + 0.011*"discrimin" + 0.010*"sentenc" + 0.010*"guilti" + 0.009*"today" + 0.009*"year"')
(2, '0.019*"offic" + 0.013*"enforc" + 0.012*"disabl" + 0.011*"state" + 0.011*"violat" + 0.011*"agreement" + 0.010*"polic" + 0.010*"said" + 0.010*"feder" + 0.010*"court" + 0.009*"general" + 0.008*"deputi" + 0.008*"communiti" + 0.008*"provid" + 0.008*"today"')


## 2.4 Add topics back to main data and explore correlation between manual labels and our estimated topics (10 points)

A. Extract the document-level topic probabilities. Within `get_document_topics`, use the argument `minimum_probability` = 0 to make sure all 3 topic probabilities are returned. Write an assert statement to make sure the length of the list is equal to the number of rows in the `doj_subset_wscores` dataframe

B. Add the topic probabilities to the `doj_subset_wscores` dataframe as columns and create a column, `top_topic`, that reflects each document to its highest-probability topic (eg topic 1, 2, or 3)

C. For each of the manual labels in `topics_clean` (Hate Crime, Civil Rights, Project Safe Childhood), print the breakdown of the % of documents with each top topic (so, for instance, Hate Crime has 246 documents-- if 123 of those documents are coded to topic_1, that would be 50%; and so on). 
**Hint**:    
pd.crosstab and normalize may be helpful: https://pandas.pydata.org/pandas-docs/version/0.23/generated/pandas.crosstab.html

D. Using a couple press releases as examples, write a 1-2 sentence interpretation of why some of the manual topics map on more cleanly to an estimated topic than other manual topic(s)

**Hint**:

- If you're getting errors, use shape, len, and other commands to check the dimensionality of things at different steps since documents may be dropped if they contain no words post-processing 

In [31]:
## your code here to get doc-level topic probabilities 

In [32]:
## your code here to add those topic probabilities to the dataframe

In [33]:
## your code here to summarize the topic proportions for each of the topics_clean 

In [34]:
## A. 
topic_probs_bydoc =[ldamod_proc.get_document_topics(item, minimum_probability=0) 
                    for item in corpus_fromdict_proc]

assert len(topic_probs_bydoc) == doj_subset_wscore.shape[0]

## B.
topic_probs_bydoc_long = pd.DataFrame([t for lst in topic_probs_bydoc for t in lst],
                                     columns = ['topic', 'probability'])

In [35]:
## add id var- we're repeating each id in the original data k times
## for the number of topics
topic_probs_bydoc_long['doc_id'] = list(np.concatenate([[one_id] * n_topics for one_id in 
                                                        doj_subset_wscore.id]).flat)

## pivot to wide format
topic_probs_bydoc_wide = pd.pivot_table(topic_probs_bydoc_long, index = ['doc_id'],
                        columns = ['topic']).reset_index().reset_index(drop = True)
topic_probs_bydoc_wide.columns = ['doc_id'] + ["topic_" + 
                                               str(i+1) for i in np.arange(0, n_topics)]
miss_ids = topic_probs_bydoc_long[~topic_probs_bydoc_long.doc_id.isin(topic_probs_bydoc_wide.doc_id)]

## merge
topic_wmeta = pd.merge(topic_probs_bydoc_wide,
                      doj_subset_wscore,
                      left_on = 'doc_id',
                      right_on = 'id')

## create indicator for listing's top topic
topic_wmeta['toptopic'] = topic_wmeta[[col for col in topic_wmeta.columns if 
                                    "topic_" in col]].idxmax(axis=1)
topic_wmeta[['topics_clean', 'toptopic']].head()

## C. breakdown
pd.crosstab(topic_wmeta.topics_clean,
           topic_wmeta.toptopic,
           normalize = "index")

## D. civil rights is more polarized; show example of topic_1 and topic_2
## one civil rights one has more overlap with hate crime topic
## one is focused more on sexual assault
topic_wmeta.contents[(topic_wmeta.toptopic == "topic_2") &
                    (topic_wmeta.topics_clean == "Civil Rights")].head(1)
topic_wmeta.contents[(topic_wmeta.toptopic == "topic_3") &
                    (topic_wmeta.topics_clean == "Civil Rights")].head(1)

,topics_clean,toptopic
0,Hate Crimes,topic_2
1,Hate Crimes,topic_2
2,Hate Crimes,topic_2
3,Hate Crimes,topic_2
4,Hate Crimes,topic_2


toptopic,topic_1,topic_2,topic_3
topics_clean,,,
Civil Rights,0.00000,0.335548,0.664452
Hate Crimes,0.00000,0.991870,0.008130
Project Safe Childhood,0.99375,0.000000,0.006250


250    The Justice Department announced today that the owner and former property manager of Perkins Parke Apartments (Perkins Parke) in Cross Lanes, West Virginia, have agreed to pay $120,000 in damages and civil penalties to settle a lawsuit alleging that three former agents sexually harassed and retaliated against female tenants in violation of the Fair Housing Act (FHA). The department’s complaint, filed in November 2014, alleged that Perkins Parke’s district manager, Anthony James, and maintenance worker, Christopher T. James, sexually harassed female tenants at the complex, and that Perkins Parke’s site manager, Kisha James, failed to take appropriate steps when tenants complained about the harassment.  The complaint alleged that the harassment included entering the residences of female tenants without permission or notice; coercing female tenants to engage in unwelcome sexual acts; making unwelcome sexual comments and unwelcome sexual advances to female tenants; subjecting female

35    The Justice Department reached a resolution agreement today with Wheaton College in Massachusetts to ensure the college implements a swift and effective response to allegations of sexual assault and harassment involving students.  In August 2015, after receiving a complaint regarding Wheaton’s handling of a report of sexual assault, the department initiated an investigation and compliance review under Title IX of the Education Amendments of 1972, which prohibits sex discrimination in education programs and activities receiving federal financial assistance.  The department reviewed Wheaton’s handling of sexual assault and harassment complaints over an approximately three and a half-year period, as well as its policies, grievance procedures, training and student education efforts. The agreement details specific steps Wheaton will take to: “We commend Wheaton College and President Hanno for their cooperation in our review as well as their work to foster a safe and healthy campus env

# 3. Extend the analysis from unigrams to bigrams (9 points)

In the previous question, you found top words via a unigram representation of the text. Now, we want to see how those top words change with bigrams (pairs of words)

A. Using the `doj_subset_wscore` data and the `processed_text` column (so the words after stemming/other preprocessing), create a column in the data called `processed_text_bigrams` that combines each consecutive pairs of word into a bigram separated by an underscore. Eg:

"depart reach settlem" would become "depart_reach reach_settlem"

Do this by writing a function `create_bigram_onedoc` that takes in a single `processed_text` string and returns a string with its bigrams structured similarly to above example
 
**Hint**: there are many ways to solve but `zip` may be helpful: https://stackoverflow.com/questions/21303224/iterate-over-all-pairs-of-consecutive-items-in-a-list

B. Print the `id`, `processed_text`, and `processed_text_bigram` columns for press release with id = 16-217

In [36]:
## your code here 

In [37]:
def create_bigrams_onedoc(one_doc):
    
    ## step 1: split
    doc_s = one_doc.split(" ")
    
    ## step 2: zip i and i+1 and combine with underscore
    big_list = [big1 + "_" + big2 for big1, big2
                 in zip(doc_s, doc_s[1:])]
    
    ## step 3: rejoin
    big_str = " ".join(big_list)
    
    ## return
    return(big_str)

In [38]:
all_big = [create_bigrams_onedoc(one_doc) 
           for one_doc in doj_subset_wscore.processed_text]

doj_subset_wscore['processed_text_bigrams'] = all_big

doj_subset_wscore.loc[doj_subset_wscore.id == "16-217",
                ['id', 'processed_text', 'processed_text_bigrams']]

,id,processed_text,processed_text_bigrams
313,16-217,reach comprehens settlement agreement citi miami miami polic resolv offic involv shoot offic announc princip deputi general vanita gupta head wifredo ferrer southern florida settlement approv miami citi commiss today effect agreement sign parti resolv claim stem offic involv shoot offic conduct violent crime control enforc find issu juli identifi pattern practic excess forc offic involv shoot violat fourth amend constitut citi complianc settlement monitor independ review former tampa florida polic chief jane castor settlement agreement citi implement comprehens reform ensur constitut polic support public trust settlement agreement design minim offic involv shoot effect quick investig offic involv shoot occur measur includ settlement repres renew commit citi miami chief rodolfo llane provid constitut polic miami resid protect public safeti sustain reform said princip deputi general gupta agreement help strengthen relationship communiti serv improv account offic fire weapon unlaw provid communiti particip enforc agreement today agreement result joint effort citi miami ensur miami polic continu effort make communiti safe protect sacr constitut citizen said ferrer oversight communic agreement seek make perman posit chang former chief orosa chief llane made applaud citi commiss vote settlement agreement build upon import reform implement citi sinc issu find includ conduct attorney staff special litig section southern florida,reach_comprehens comprehens_settlement settlement_agreement agreement_citi citi_miami miami_miami miami_polic polic_resolv resolv_offic offic_involv involv_shoot shoot_offic offic_announc announc_princip princip_deputi deputi_general general_vanita vanita_gupta gupta_head head_wifredo wifredo_ferrer ferrer_southern southern_florida florida_settlement settlement_approv approv_miami miami_citi citi_commiss commiss_today today_effect effect_agreement agreement_sign sign_parti parti_resolv resolv_claim claim_stem stem_offic offic_involv involv_shoot shoot_offic offic_conduct conduct_violent violent_crime crime_control control_enforc enforc_find find_issu issu_juli juli_identifi identifi_pattern pattern_practic practic_excess excess_forc forc_offic offic_involv involv_shoot shoot_violat violat_fourth fourth_amend amend_constitut constitut_citi citi_complianc complianc_settlement settlement_monitor monitor_independ independ_review review_former former_tampa tampa_florida florida_polic polic_chief chief_jane jane_castor castor_settlement settlement_agreement agreement_citi citi_implement implement_comprehens comprehens_reform reform_ensur ensur_constitut constitut_polic polic_support support_public public_trust trust_settlement settlement_agreement agreement_design design_minim minim_offic offic_involv involv_shoot shoot_effect effect_quick quick_investig investig_offic offic_involv involv_shoot shoot_occur occur_measur measur_includ includ_settlement settlement_repres repres_renew renew_commit commit_citi citi_miami miami_chief chief_rodolfo rodolfo_llane llane_provid provid_constitut constitut_polic polic_miami miami_resid resid_protect protect_public public_safeti safeti_sustain sustain_reform reform_said said_princip princip_deputi deputi_general general_gupta gupta_agreement agreement_help help_strengthen strengthen_relationship relationship_communiti communiti_serv serv_improv improv_account account_offic offic_fire fire_weapon weapon_unlaw unlaw_provid provid_communiti communiti_particip particip_enforc enforc_agreement agreement_today today_agreement agreement_result result_joint joint_effort effort_citi citi_miami miami_ensur ensur_miami miami_polic polic_continu continu_effort effort_make make_communiti communiti_safe safe_protect protect_sacr sacr_constitut constitut_citizen citizen_said said_ferrer ferrer_oversight oversight_communic communic_agreement agreement_seek seek_make make_perman perman_posit posit_chang chang_former former_chief chief_orosa oro

C. Use the create_dtm function and the `processed_text_bigrams` column to create a document-term matrix (`dtm_bigram`) with these bigrams. Keep the following three columns in the data: `id`, `topics_clean`, and `compound` 

D. Print the 
 (1) dimensions of the `dtm` matrix from question 2.2  and 
 (2) the dimensions of the `dtm_bigram` matrix. Comment on why the bigram matrix has more dimensions than the unigram matrix 

In [39]:
dtm_bigram = create_dtm(doj_subset_wscore.processed_text_bigrams,
                doj_subset_wscore[['id', 'topics_clean', 'compound']].rename(columns = 
                            {'id': 'id_raw',
                            'topics_clean': 'topics_clean_raw',
                            'compound': 'compound_raw'}))

print("dim of unigram dtm is: ")
dtm.shape
print("dim of bigram dtm is: ")
dtm_bigram.shape

dim of unigram dtm is: 


(717, 6870)

dim of bigram dtm is: 


(717, 72721)

E. Find and print the 10 most prevelant bigrams for each of the three topics_clean using the `get_topwords` function from 2.2


In [40]:
for topic in dtm_bigram.topics_clean_raw.unique():

    subset = dtm_bigram[dtm_bigram.topics_clean_raw == topic]
    print("For topic: " + topic)
    get_topw(subset)

For topic: Civil Rights


fair_hous         231
princip_deputi    221
deputi_general    221
vanita_gupta      202
gupta_head        200
general_vanita    199
said_princip      186
unit_state        156
nation_origin     143
consent_decre     128
dtype: int64

For topic: Project Safe Childhood


safe_childhood       474
project_safe         472
child_pornographi    450
child_exploit        281
sexual_exploit       223
exploit_children     200
plead_guilti         197
exploit_obscen       176
obscen_section       175
child_sexual         174
dtype: int64

For topic: Hate Crimes


hate_crime          379
african_american    367
plead_guilti        275
year_prison         161
special_agent       124
racial_motiv        114
thoma_perez         111
grand_juri          101
perez_general        95
said_thoma           91
dtype: int64

# 4. Optional extra credit 1 (1 point)

You notice that the pharmaceutical kickbacks press release we analyzed in question 1 was for an indictment, and that in the original data, there's not a clear label for whether a press release outlines an indictment (charging someone with a crime), a conviction (convicting them after that charge either via a settlement or trial), or a sentencing (how many years of prison or supervised release a defendant is sentenced to after their conviction).

You want to see if you can identify pairs of press releases where one press release is from one stage (e.g., indictment) and another is from a different stage (e.g., a sentencing).

You decide that one way to approach is to find the pairwise string similarity between each of the processed press releases in `doj_subset`. There are many ways to do this, so Google for some approaches, focusing on ones that work well for entire documents rather than small strings.

Find the top two pairs (so four press releases total)-- do they seem like different stages of the same crime or just press releases covering similar crimes?

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer

doj_subset_wscore.index = np.arange(0, doj_subset_wscore.shape[0])
vect = TfidfVectorizer(min_df=1, stop_words=list_stopwords)                                                                                                                                                                                                   
tfidf = vect.fit_transform(doj_subset_wscore.processed_text)                                                                                                                                                                                                                       


In [42]:
pairwise_similarity = tfidf * tfidf.T 

p_arr = pairwise_similarity.toarray()         
np.fill_diagonal(p_arr, np.nan)  

p_df = pd.DataFrame(p_arr)

p_df.columns = ["release_" + str(i) for i in np.arange(0, p_df.shape[0])]
p_df["closest_pair_which"] = p_df[[col for col in p_df.columns if 
                        "release_" in col]].idxmax(axis=1)

all_val = list()
for i in range(0, p_df.shape[0]):
    
    top_col = p_df.loc[i, 'closest_pair_which']
    val_topcol = p_df.loc[i, top_col]
    all_val.append(val_topcol)
    
p_df['closest_pair_val'] = all_val

In [43]:
## p_df['release'] = ["release_" + str(i) for i in p_df.index]

p_df.sort_values(by = 'closest_pair_val', ascending = False).head(4)



,release_0,release_1,release_2,release_3,release_4,release_5,release_6,release_7,release_8,release_9,...,release_709,release_710,release_711,release_712,release_713,release_714,release_715,release_716,closest_pair_which,closest_pair_val
203,0.044717,0.171360,0.195996,0.181407,0.019839,0.027909,0.212223,0.199620,0.017313,0.176153,...,0.034923,0.050818,0.036609,0.039692,0.158053,0.144640,0.064152,0.034154,release_686,0.987779
686,0.045696,0.173962,0.208221,0.189667,0.022757,0.032026,0.219417,0.206318,0.018997,0.172282,...,0.036583,0.046796,0.038695,0.040713,0.164003,0.147468,0.065110,0.025660,release_203,0.987779
519,0.039916,0.213964,0.234971,0.266846,0.030011,0.017526,0.288439,0.245270,0.035201,0.209583,...,0.047712,0.042335,0.053765,0.049158,0.203080,0.161042,0.060306,0.042736,release_521,0.969668
521,0.038933,0.202440,0.220400,0.245330,0.030099,0.016295,0.272100,0.238373,0.033364,0.203563,...,0.044514,0.045139,0.051351,0.047144,0.194769,0.152854,0.057735,0.049794,release_519,0.969668


In [44]:


doj_subset_wscore.contents[doj_subset_wscore.index.isin([203, 686])]
doj_subset_wscore.contents[doj_subset_wscore.index.isin([521, 519])]

203    A Church Hill, Maryland, resident was sentenced today to 20 years in prison to be followed by a lifetime term of supervised release for enticement of a minor to engage in sexual activity and attempting to transfer obscene materials to a minor, announced Acting Assistant Attorney General Kenneth A. Blanco of the Justice Department’s Criminal Division and Acting U.S. Attorney Benjamin G. Greenberg of the Southern District of Florida.    Lee Robert Moore, 38, pleaded guilty March 1, 2017, before U.S. District Judge Daniel T. K. Hurley of the Southern District of Florida. Moore was employed by the U.S. Secret Service-Uniformed Division and was assigned to the White House at the time of his arrest on Nov. 9, 2015, and has remained in custody since that time. Moore has since been terminated from his Secret Service position.   According to admissions made in connection with his plea, Moore maintained a profile on the social media application “Meet24,” which provides a mobile-based plat

519                                                                                                                                                 A 25-year-old man who coerced sexually explicit photos and videos from a minor and distributed them on the Internet pleaded guilty on Oct. 2, to coercion and enticement of a minor to engage in sexual activity.    Acting Assistant Attorney General Kenneth A. Blanco of the Justice Department’s Criminal Division and Acting U.S. Attorney David C. Weiss of the District of Delaware made the announcement. Justin R. Gulisano of Newark, New York, was charged on March 10, 2016, and pleaded guilty before U.S. District Judge Leonard P. Stark of the District of Delaware.   According to admissions made in connection with the plea agreement, Gulisano met the victim online when the victim was 15 years old.  Gulisano began requesting and received sexually explicit images and videos from the victim.  Gulisano posted the sexually explicit videos of the victim